# Analysis of Macro Ecological and Economical factors on Infrastructure Distribution in the EU

**Team members:**
- FirstName FamilyName
- FirstName FamilyName
- FirstName FamilyName
- FirstName FamilyName
---

## Dataset Description

This project investigates how physical geography (terrain difficulty and proximity to navigable water) alongside public investment budgets, explains the distribution of road and rail infrastructure across EU NUTS2 regions.

We combine five data sources, all joined on the NUTS2 regional classification:

| Source | Variable(s) | Type | Temporal coverage |
|---|---|---|---|
| Eurostat `tran_r_net` | Road + rail density (km/1000km²) | Time series | ~1990–2022 |
| Eurostat `tgs00024` | Population density | Time series | 1990–2023 |
| Eurostat `gov_10a_exp` | Regional public expenditure | Time series | 1995–2023 |
| NASA SRTM (via `elevation`) | Mean elevation + terrain slope | Static raster | 2000 snapshot |
| HydroSHEDS HydroRIVERS | Distance to nearest major river | Static vector | Current |

**Research question:** Do terrain difficulty, proximity to water, and public budget explain infrastructure density across EU regions and can past values predict future infrastructure levels?

**Dimensions:**
- *Temporal*: Eurostat time series (1995–2022)
- *Spatial*: NUTS2 polygons (~240 EU regions)
- *Analytical*: Infrastructure density as a function of geographic and fiscal variables

## 0. Installations

Run once per environment. Comment out after first run.

In [45]:
# !pip install -r requirements.txt --prefer-binary 

## 1. Imports

In [46]:
import os
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import pandas as pd
import numpy as np

# Geospatial
import geopandas as gpd
from shapely.geometry import Point
import rasterstats
import rasterio
import boto3

# Eurostat API
import eurostat

# Visualisation
import matplotlib.pyplot as plt
import plotly.express as px

print("All imports OK")

All imports OK


## 2. Data Loading

> GenAI Note: This section has been written with the help of Generative AI, who has been prompted to join all datasets on NUTS2 region code and generate the topology based on those, too. We still do understand the code, but did not write it by hand. It was only one prompt, zero-shot with no examples. We also generated a notebook skeleton to help make presentation more readable quickly and focus on analysis.

Each function loads one data source and returns it in a clean, NUTS2-joinable format.

> **Note:** HydroRIVERS must be downloaded manually once and placed in `./data/`.
> - HydroRIVERS Europe: Should be in repo but just in case - https://www.hydrosheds.org/products/hydrorivers → download `HydroRIVERS_v10_eu.shp`
> - Terrain (Copernicus GLO-30) streams automatically from AWS S3 on first run, then cached to `./data/terrain_nuts2.csv` - should also be in repo

In [1]:
# ─────────────────────────────────────────────
# Function: load_nuts2
# Input:  resolution string (e.g. '20M'), year int (e.g. 2021)
# Output: GeoDataFrame of NUTS2 polygons with columns [NUTS_ID, geometry]
# ─────────────────────────────────────────────
def load_nuts2(resolution='20M', year=2021):
    url = (
        f"https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/"
        f"NUTS_RG_{resolution}_{year}_4326.geojson"
    )
    gdf = gpd.read_file(url)
    # Keep only NUTS level 2
    gdf = gdf[gdf['LEVL_CODE'] == 2][['NUTS_ID', 'NUTS_NAME', 'geometry']].copy()
    gdf = gdf.reset_index(drop=True)
    print(f"NUTS2 loaded: {len(gdf)} regions")
    return gdf

In [48]:
# ─────────────────────────────────────────────
# Function: load_infrastructure
# Input:  none
# Output: DataFrame with columns [NUTS_ID, year, motorway_density, rail_density]
#         density in km / 1000 km² of region area
# Source: Eurostat dataset tran_r_net
# ─────────────────────────────────────────────
def load_infrastructure():
    df = eurostat.get_data_df('tran_r_net')

    # tran_r_net has columns: freq, unit, vehicle, geo\TIME_PERIOD, then year columns
    # Melt from wide to long format
    id_vars = [c for c in df.columns if not str(c).isdigit() and '\\' not in str(c)]
    geo_col = [c for c in df.columns if '\\' in str(c)][0]

    df = df.rename(columns={geo_col: 'NUTS_ID'})
    year_cols = [c for c in df.columns if str(c).isdigit()]

    df_long = df.melt(
        id_vars=[c for c in df.columns if c not in year_cols],
        value_vars=year_cols,
        var_name='year',
        value_name='infra_km'
    )
    df_long['year'] = df_long['year'].astype(int)
    df_long['infra_km'] = pd.to_numeric(df_long['infra_km'], errors='coerce')

    # Keep only NUTS2 (4-character codes like FR10, DE21)
    df_long = df_long[df_long['NUTS_ID'].str.len() == 4]

    print(f"Infrastructure data: {df_long.shape[0]} rows, years {df_long['year'].min()}–{df_long['year'].max()}")
    return df_long

In [49]:
# ─────────────────────────────────────────────
# Function: load_population
# Input:  none
# Output: DataFrame with columns [NUTS_ID, year, pop_density]
#         pop_density in inhabitants / km²
# Source: Eurostat dataset tgs00024
# ─────────────────────────────────────────────
def load_population():
    df = eurostat.get_data_df('tgs00024')

    geo_col = [c for c in df.columns if '\\' in str(c)][0]
    df = df.rename(columns={geo_col: 'NUTS_ID'})
    year_cols = [c for c in df.columns if str(c).isdigit()]

    df_long = df.melt(
        id_vars=[c for c in df.columns if c not in year_cols],
        value_vars=year_cols,
        var_name='year',
        value_name='pop_density'
    )
    df_long['year'] = df_long['year'].astype(int)
    df_long['pop_density'] = pd.to_numeric(df_long['pop_density'], errors='coerce')
    df_long = df_long[df_long['NUTS_ID'].str.len() == 4]

    print(f"Population data: {df_long.shape[0]} rows, years {df_long['year'].min()}–{df_long['year'].max()}")
    return df_long

In [50]:
# ─────────────────────────────────────────────
# Function: load_budget
# Input:  none
# Output: DataFrame with columns [NUTS_ID, year, expenditure_mEUR]
# Source: Eurostat dataset gov_10a_exp (general government expenditure by function)
# Note:   We filter for transport infrastructure expenditure (COFOG code GF0452)
# ─────────────────────────────────────────────
def load_budget():
    df = eurostat.get_data_df('gov_10a_exp')

    geo_col = [c for c in df.columns if '\\' in str(c)][0]
    df = df.rename(columns={geo_col: 'NUTS_ID'})

    # Filter to transport expenditure if cofog column exists
    if 'cofog99' in df.columns:
        df = df[df['cofog99'].str.startswith('GF04', na=False)]

    year_cols = [c for c in df.columns if str(c).isdigit()]
    df_long = df.melt(
        id_vars=[c for c in df.columns if c not in year_cols],
        value_vars=year_cols,
        var_name='year',
        value_name='expenditure_mEUR'
    )
    df_long['year'] = df_long['year'].astype(int)
    df_long['expenditure_mEUR'] = pd.to_numeric(df_long['expenditure_mEUR'], errors='coerce')

    # This dataset is country-level (NUTS0 = 2-char codes), keep as-is
    # We will join on country prefix when merging with NUTS2
    df_long['country'] = df_long['NUTS_ID'].str[:2]

    print(f"Budget data: {df_long.shape[0]} rows, years {df_long['year'].min()}–{df_long['year'].max()}")
    return df_long

In [51]:
# ─────────────────────────────────────────────
# Function: load_terrain
# Input:  nuts2_gdf : GeoDataFrame of NUTS2 polygons (EPSG:4326)
#         output_csv: path to save/load precomputed results
# Output: DataFrame with columns [NUTS_ID, mean_elevation_m, elevation_std]
#         elevation_std is used as a terrain roughness / difficulty proxy
#
# Source: Copernicus GLO-30 DEM (30m resolution) hosted on AWS S3
#         No account or GDAL CLI tools required — uses rasterio (anonymous S3)
#         Tiles are streamed directly, no full EU download needed.
#
# How it works:
#   Copernicus GLO-30 is tiled as 1°×1° GeoTIFFs named by lat/lon.
#   We find which tiles overlap each NUTS2 bounding box, stream them from S3,
#   and compute mean elevation + std as a terrain roughness proxy.
#
# First run: streams tiles from S3, takes 15–40 min depending on connection.
# Subsequent runs: loads from output_csv instantly.
# ─────────────────────────────────────────────

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import rasterio.merge
from rasterio.windows import from_bounds

COP_DEM_BUCKET = "copernicus-dem-30m"


def _tile_s3_key(lat, lon):
    """
    Build the S3 key for a Copernicus GLO-30 tile given its SW corner.
    Input:  lat (int), lon (int) — SW corner of 1°×1° tile
    Output: S3 key string
    """
    lat_str = f"{'N' if lat >= 0 else 'S'}{abs(lat):02d}"
    lon_str = f"{'E' if lon >= 0 else 'W'}{abs(lon):03d}"
    name = f"Copernicus_DSM_COG_10_{lat_str}_00_{lon_str}_00_DEM"
    return f"{name}/{name}.tif"


def _get_tiles_for_bounds(bounds):
    """
    Return list of (lat, lon) SW corners for all 1°×1° tiles overlapping bounds.
    Input:  bounds = (minx, miny, maxx, maxy) in EPSG:4326
    Output: list of (lat, lon) integer tuples
    """
    minx, miny, maxx, maxy = bounds
    tiles = []
    for lat in range(int(np.floor(miny)), int(np.ceil(maxy))):
        for lon in range(int(np.floor(minx)), int(np.ceil(maxx))):
            tiles.append((lat, lon))
    return tiles


def load_terrain(nuts2_gdf, output_csv='./data/terrain_nuts2.csv'):
    """
    Compute mean elevation and terrain roughness per NUTS2 region
    using Copernicus GLO-30 DEM streamed from public AWS S3.

    Input:  nuts2_gdf  — GeoDataFrame of NUTS2 polygons (EPSG:4326)
            output_csv — path to cache results; skip computation if exists
    Output: DataFrame [NUTS_ID, mean_elevation_m, elevation_std]
    """
    os.makedirs('./data', exist_ok=True)

    # Load from cache if available (recommended after first run)
    if os.path.exists(output_csv):
        print(f"Loading cached terrain data from {output_csv}")
        return pd.read_csv(output_csv)

    print("Computing terrain stats from Copernicus GLO-30 (streams from AWS S3)...")
    print("This runs once and caches to CSV. Expect 15–40 min on first run.")

    nuts2_proj = nuts2_gdf[['NUTS_ID', 'geometry']].to_crs('EPSG:4326').copy()

    results = []

    for _, region in nuts2_proj.iterrows():
        nuts_id = region['NUTS_ID']
        bounds = region.geometry.bounds  # (minx, miny, maxx, maxy)
        tiles_needed = _get_tiles_for_bounds(bounds)

        tile_arrays = []

        for (lat, lon) in tiles_needed:
            key = _tile_s3_key(lat, lon)
            s3_url = f"s3://{COP_DEM_BUCKET}/{key}"

            try:
                with rasterio.Env(
                    aws_unsigned=True,
                    GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR'
                ):
                    with rasterio.open(s3_url) as src:
                        # Read only the window overlapping the region bbox
                        window = from_bounds(*bounds, transform=src.transform)
                        data = src.read(1, window=window)
                        tile_arrays.append(data)
            except Exception:
                # Tile doesn't exist (ocean, missing coverage) — skip silently
                continue

        if not tile_arrays:
            results.append({
                'NUTS_ID': nuts_id,
                'mean_elevation_m': np.nan,
                'elevation_std': np.nan
            })
            continue

        # Flatten all tile pixels for this region into one array
        all_pixels = np.concatenate([a.flatten() for a in tile_arrays])
        # Mask nodata values (GLO-30 nodata = -32768, ocean = 0)
        all_pixels = all_pixels[(all_pixels > -9000) & (all_pixels < 9000)]

        results.append({
            'NUTS_ID': nuts_id,
            'mean_elevation_m': float(np.mean(all_pixels)) if len(all_pixels) > 0 else np.nan,
            'elevation_std': float(np.std(all_pixels)) if len(all_pixels) > 0 else np.nan
        })

        print(f"  ✓ {nuts_id} — elev: {results[-1]['mean_elevation_m']:.0f}m, "
              f"roughness std: {results[-1]['elevation_std']:.0f}m")

    terrain_df = pd.DataFrame(results)
    terrain_df.to_csv(output_csv, index=False)
    print(f"\nTerrain data saved to {output_csv}")
    return terrain_df

In [52]:
# ─────────────────────────────────────────────
# Function: load_water_proximity
# Input:  nuts2_gdf   : GeoDataFrame of NUTS2 polygons (EPSG:4326)
#         rivers_path : path to HydroRIVERS Europe geodatabase zip
# Output: DataFrame with columns [NUTS_ID, dist_to_river_km]
#         dist = distance from region centroid to nearest major river segment
# Source: HydroSHEDS HydroRIVERS — https://www.hydrosheds.org/products/hydrorivers
#         File needed: HydroRIVERS_v10_eu_gdb.zip → place in ./data/
# ─────────────────────────────────────────────
def load_water_proximity(nuts2_gdf, rivers_path='./data/HydroRIVERS_v10_eu_gdb.zip'):
    if not os.path.exists(rivers_path):
        raise FileNotFoundError(
            f"HydroRIVERS geodatabase not found at {rivers_path}.\n"
            "Download from: https://www.hydrosheds.org/products/hydrorivers\n"
            "Place HydroRIVERS_v10_eu_gdb.zip into ./data/"
        )

    print("Loading HydroRIVERS from geodatabase zip...")
    rivers = gpd.read_file(f"zip://{rivers_path}", layer='HydroRIVERS_v10_eu')

    # Keep only major rivers (Strahler order >= 5) to match historical navigation logic
    if 'ORD_STRA' in rivers.columns:
        rivers = rivers[rivers['ORD_STRA'] >= 5]

    # Work in a metric CRS for accurate distance calculation
    crs_metric = 'EPSG:3035'  # ETRS89-LAEA, standard for EU spatial analysis
    nuts2_metric = nuts2_gdf.to_crs(crs_metric)
    rivers_metric = rivers.to_crs(crs_metric)

    # Compute centroid of each NUTS2 region
    nuts2_metric['centroid'] = nuts2_metric.geometry.centroid

    # Union all river lines into one geometry for faster distance computation
    rivers_union = rivers_metric.geometry.unary_union

    # Distance from each centroid to nearest river (in metres → convert to km)
    nuts2_metric['dist_to_river_km'] = nuts2_metric['centroid'].apply(
        lambda pt: pt.distance(rivers_union) / 1000
    )

    result = nuts2_metric[['NUTS_ID', 'dist_to_river_km']].copy()
    print(f"Water proximity computed for {len(result)} NUTS2 regions.")
    return result

## 3. Data Exploration

Before joining, we inspect each source individually to understand structure, missing values, and value ranges.

In [53]:
# ─────────────────────────────────────────────
# Function: explore_dataframe
# Input:  df: any DataFrame, name: string label for display
# Output: prints shape, dtypes, missing values, and numeric ranges
# ─────────────────────────────────────────────
def explore_dataframe(df, name="Dataset"):
    print(f"\n{'='*50}")
    print(f" {name}")
    print(f"{'='*50}")
    print(f"Shape:          {df.shape[0]} rows × {df.shape[1]} columns")
    print(f"\nColumn types:")
    print(df.dtypes.to_string())
    print(f"\nMissing values per column:")
    missing = df.isnull().sum()
    print(missing[missing > 0].to_string() if missing.any() else "  None")
    print(f"\nNumeric ranges:")
    print(df.describe().to_string())

## 4. Join Everything on NUTS2

This produces the master dataframe used for all downstream analysis.

In [54]:
# ─────────────────────────────────────────────
# Function: build_master_df
# Input:  all loaded dataframes + nuts2 geodataframe
# Output: merged GeoDataFrame with all variables joined on NUTS_ID (+ year for time series)
#
# Structure of output:
#   NUTS_ID | year | infra_km | pop_density | expenditure_mEUR |
#   mean_elevation_m | elevation_std | dist_to_river_km | geometry
#
# Static variables (terrain, water) are repeated across all years per region.
# Budget is country-level (NUTS0) and broadcast to all NUTS2 in that country.
# ─────────────────────────────────────────────
def build_master_df(nuts2_gdf, infra_df, pop_df, budget_df, terrain_df, water_df):

    # Start from infrastructure as the temporal backbone
    df = infra_df[['NUTS_ID', 'year', 'infra_km']].copy()

    # Join population density (same NUTS_ID + year key)
    df = df.merge(
        pop_df[['NUTS_ID', 'year', 'pop_density']],
        on=['NUTS_ID', 'year'], how='left'
    )

    # Join budget on country code (broadcast NUTS0 → NUTS2)
    df['country'] = df['NUTS_ID'].str[:2]
    budget_agg = budget_df.groupby(['country', 'year'])['expenditure_mEUR'].sum().reset_index()
    df = df.merge(budget_agg, on=['country', 'year'], how='left')

    # Join static terrain features (no year dimension)
    df = df.merge(terrain_df[['NUTS_ID', 'mean_elevation_m', 'elevation_std']], on='NUTS_ID', how='left')

    # Join static water proximity
    df = df.merge(water_df[['NUTS_ID', 'dist_to_river_km']], on='NUTS_ID', how='left')

    # Attach geometries for spatial analysis
    gdf = nuts2_gdf[['NUTS_ID', 'NUTS_NAME', 'geometry']].merge(df, on='NUTS_ID', how='right')
    gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs='EPSG:4326')

    print(f"Master dataframe built: {gdf.shape[0]} rows × {gdf.shape[1]} columns")
    print(f"NUTS2 regions: {gdf['NUTS_ID'].nunique()}")
    print(f"Years: {gdf['year'].min()}–{gdf['year'].max()}")
    print(f"Missing values:\n{gdf.isnull().sum()[gdf.isnull().sum() > 0]}")
    return gdf

## 5. Main

Runs the full pipeline and stores the master dataframe for use in analysis cells.

In [55]:
import fiona
import os
path = os.path.abspath("./data/HydroRIVERS_v10_eu_gdb.zip")
print(path)  # sanity check it resolves correctly
fiona.listlayers(path)

c:\Users\ricar\Desktop\ESILV\dataanalysis\project\data\HydroRIVERS_v10_eu_gdb.zip


['HydroRIVERS_v10_eu']

In [ ]:
if __name__ == "__main__":

    # ── Step 1: Load spatial backbone ──────────────────────────
    nuts2 = load_nuts2()

    # ── Step 2: Load time series from Eurostat ──────────────────
    infra  = load_infrastructure()
    pop    = load_population()
    budget = load_budget()

    # ── Step 3: Load static geographic features ─────────────────
    terrain = load_terrain(nuts2)
    water   = load_water_proximity(nuts2)

    # ── Step 4: Explore each source before joining ───────────────
    explore_dataframe(infra,   "Infrastructure (tran_r_net)")
    explore_dataframe(pop,     "Population density (tgs00024)")
    explore_dataframe(budget,  "Public expenditure (gov_10a_exp)")
    explore_dataframe(terrain, "Terrain (SRTM)")
    explore_dataframe(water,   "Water proximity (HydroRIVERS)")

    # ── Step 5: Build master dataframe ───────────────────────────
    master = build_master_df(nuts2, infra, pop, budget, terrain, water)

    # ── Step 6: Quick sanity plot ─────────────────────────────────
    fig, ax = plt.subplots(1, 1, figsize=(14, 10))
    master[master['year'] == 2019].plot(
        column='infra_km',
        ax=ax,
        legend=True,
        cmap='YlOrRd',
        missing_kwds={'color': 'lightgrey'},
        legend_kwds={'label': 'Infrastructure density (km/1000km²)'}
    )
    ax.set_title('EU Infrastructure Density by NUTS2 Region (2019)', fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    plt.savefig('./data/sanity_check_map.png', dpi=150)
    plt.show()

    print("\nPipeline complete. `master` dataframe is ready for analysis.")
    print(master.head())

NUTS2 loaded: 334 regions
Infrastructure data: 83965 rows, years 1990–2024
Population data: 4080 rows, years 2013–2024
Budget data: 5753556 rows, years 1990–2025
Computing terrain stats from Copernicus GLO-30 (streams from AWS S3)...
This runs once and caches to CSV. Expect 15–40 min on first run.
  ✓ CZ05 — elev: 417m, roughness std: 169m
  ✓ CZ06 — elev: 419m, roughness std: 154m
  ✓ CZ07 — elev: 394m, roughness std: 184m
  ✓ CZ08 — elev: 402m, roughness std: 188m
  ✓ DE11 — elev: 393m, roughness std: 121m
  ✓ DE12 — elev: 351m, roughness std: 200m
  ✓ DE13 — elev: 546m, roughness std: 254m
  ✓ DE14 — elev: 608m, roughness std: 138m
  ✓ DE21 — elev: 627m, roughness std: 328m
  ✓ DE22 — elev: 502m, roughness std: 177m
  ✓ DE23 — elev: 496m, roughness std: 114m
  ✓ DE24 — elev: 451m, roughness std: 121m
  ✓ DE25 — elev: 426m, roughness std: 80m
  ✓ DE26 — elev: 333m, roughness std: 100m
  ✓ DE27 — elev: 697m, roughness std: 338m
  ✓ DE30 — elev: 50m, roughness std: 13m
  ✓ DEC0 — elev:

In [ ]:
master.isnull().mean().mul(100).round(2).astype(str) + '%'

NUTS_ID               0.0%
NUTS_NAME            8.46%
geometry             8.46%
year                  0.0%
infra_km            48.58%
pop_density         69.66%
country               0.0%
expenditure_mEUR    15.67%
mean_elevation_m    92.12%
elevation_std       92.12%
dist_to_river_km     8.46%
dtype: str

---

## 6. Analysis (to be completed by the team)

Add your query functions here. Each one should:
- Have a docstring with Input / Output
- Include a markdown cell explaining what the indicator means and how to interpret it
- Be called from a `main` block at the end

### Query 1 — Grouping [1pt]
*Example: which NUTS2 regions have the highest infrastructure density per unit of public expenditure?*

### Query 2 — Pattern Mining [2pt]
*Example: frequent itemsets of (terrain_bucket, water_bucket, budget_bucket) → infrastructure level clusters*

### Query 3 — Spatial [2pt]
*Example: spatial clustering (DBSCAN or KMeans on geography) of under/over-infrastructure regions relative to geographic difficulty*

### Query 4 — Temporal [2pt]
*Example: time series forecasting of infrastructure density using lag features (infra_t-1, budget_t-1, pop_t-1)*